# GoEmotions dataset exploration

This notebook is **exploratory only** -- it inspects the raw and filtered
GoEmotions data to sanity-check the shared pipeline in `src/emotion_ait/`.
It does not train or evaluate any model (see `mdl_classifier.py` /
`lstm_classifier.py` for those stubs).

Run `python scripts/prepare_data.py` once before using this notebook, so
that `data/raw/` and `data/processed/` are populated.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import pandas as pd

from emotion_ait.config import load_config
from emotion_ait.data_loader import load_raw_dataset
from emotion_ait.preprocessing import class_counts_per_split

config = load_config()
config

ModuleNotFoundError: No module named 'emotion_ait'

## Raw dataset: shape, columns, and a few example rows

In [ ]:
raw_frames = load_raw_dataset(config)
for split, df in raw_frames.items():
    print(split, df.shape, list(df.columns))
raw_frames["train"].head()

## How many raw comments are single-label vs. multi-label?

In [ ]:
label_count_hist = raw_frames["train"]["labels"].apply(len).value_counts().sort_index()
label_count_hist

## Processed (filtered, 4-class) dataset: retained counts per split/class

In [ ]:
processed_frames = {
    split: pd.read_csv(config.paths.processed_dir / f"{split}.csv") for split in config.dataset.splits
}
processed_df = pd.concat(processed_frames.values(), ignore_index=True)
class_counts_per_split(processed_df)

## Quick look at text length distribution per target class (train split)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

train_df = processed_frames["train"].copy()
train_df["n_words"] = train_df["text"].str.split().apply(len)

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=train_df, x="label_name", y="n_words", ax=ax)
ax.set_title("Word count per comment, by target emotion (train split)")
plt.show()

## Example comments per target class

In [ ]:
for label_name in config.target_task.target_emotions:
    print(f"--- {label_name} ---")
    sample = train_df[train_df["label_name"] == label_name]["text"].sample(3, random_state=42)
    for text in sample:
        print(" -", text)
    print()